In [ ]:
# ================================================================
#  02_assemble — Ghép cặp human/AI + gán split + QA gate G1/G4/G5
#  Input : output/human_clean.parquet, output/ai_clean.parquet (01_clean_qa.ipynb)
#  Output: dataset_{train,val,test_ood_model}_normalized.parquet (paired,
#          đi tiếp qua attack-dataset.ipynb/paraphrase-attack.ipynb/llm-attack.ipynb,
#          rồi 02b_deconfound.ipynb -> 03_diagnostics.ipynb (cổng G6))
#          dataset_test_fp_normalized.parquet (KHÔNG qua attack — đi thẳng
#          tới 04_melt_for_training.ipynb, vì test_fp đo false-positive,
#          không phải robustness)
# ================================================================

import hashlib
import re
from pathlib import Path

import numpy as np
import pandas as pd

# ── CELL 1: Config ──────────────────────────────────────────────
try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path(".").resolve()

OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HUMAN_CLEAN_PATH = OUTPUT_DIR / "human_clean.parquet"
AI_CLEAN_PATH = OUTPUT_DIR / "ai_clean.parquet"

TRAIN_VAL_RATIO = 0.94  # trong train_val_pool: train 16,000 : val 1,000 ~ 94:6
TRAIN_FAMILY_QUOTA = 0.25  # phải khớp TRAIN_FAMILY_QUOTA trong genai.ipynb

# Phai khop CHINH XAC SPLIT_POOL_WEIGHTS trong genai.ipynb -- dung de tinh lai
# split_pool cua tung paper_id (deterministic theo hash, khong can doc lai state
# tu genai.ipynb). Dung o CELL 5 de loc leftover_human dung pool, tranh leak G1.
SPLIT_POOL_WEIGHTS = [("train_val", 70), ("ood_model", 15), ("test_fp", 15)]


def _hash01(key):
    digest = hashlib.sha256(key.encode("utf-8")).hexdigest()
    return int(digest[:8], 16) / 0xFFFFFFFF


def assign_train_val(paper_id):
    return "train" if _hash01(f"train_val_split::{paper_id}") < TRAIN_VAL_RATIO else "val"


def assign_split_pool(paper_id):
    """Ban sao chinh xac cua ham cung ten trong genai.ipynb (cung cong thuc hash) --
    de xac dinh 1 paper_id THUC SU thuoc pool nao, thay vi suy doan tu viec no co
    duoc ghep cap AI hay khong (nguon goc bug G1 leak khi generation chua chay het)."""
    r = _hash01(f"split_pool::{paper_id}") * 100
    cum = 0.0

    for label, w in SPLIT_POOL_WEIGHTS:
        cum += w

        if r <= cum:
            return label

    return SPLIT_POOL_WEIGHTS[-1][0]


# Feature bề mặt dùng để cân bằng confound (spec §8) — PHẢI khớp định nghĩa trong
# 03_diagnostics.ipynb để cái mình cân bằng đúng là cái diagnostics đo.
_SENT_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")


def avg_sentence_len(text):
    text = str(text)
    n_tokens = max(1, len(text.split()))
    sentences = [s for s in _SENT_SPLIT_RE.split(text.strip()) if s]
    return n_tokens / max(1, len(sentences))


print(f"HUMAN_CLEAN_PATH : {'✅' if HUMAN_CLEAN_PATH.exists() else '❌'}  {HUMAN_CLEAN_PATH}")
print(f"AI_CLEAN_PATH    : {'✅' if AI_CLEAN_PATH.exists() else '❌'}  {AI_CLEAN_PATH}")


# ── CELL 2: Load ───────────────────────────────────────────────
human_df = pd.read_parquet(HUMAN_CLEAN_PATH)
ai_df = pd.read_parquet(AI_CLEAN_PATH)

print(f"human_clean : {len(human_df):,} dòng")
print(f"ai_clean    : {len(ai_df):,} dòng ({ai_df['text_type'].value_counts().to_dict()})")

ai_rewrite = ai_df[ai_df["text_type"] == "ai"].copy()
ai_polish = ai_df[ai_df["text_type"] == "ai_polish"].copy()


# ── CELL 3: Ghép cặp (paper_id, source) — wide format ────────────
def make_pairs(human_df, ai_sub_df):
    merged = human_df.merge(
        ai_sub_df,
        on=["paper_id", "source"],
        how="inner",
        suffixes=("_human", "_ai"),
    )

    merged = merged.rename(columns={
        "text_human": "human_text",
        "text_ai": "ai_text",
        "word_count_ai": "word_count",
    })

    merged["sample_id"] = merged["paper_id"].astype(str) + "_" + merged["source"].astype(str)

    return merged


rewrite_pairs = make_pairs(human_df, ai_rewrite)
polish_pairs = make_pairs(human_df, ai_polish)

print(f"\nRewrite pairs (human+ai): {len(rewrite_pairs):,}")
print(f"Polish pairs  (human+ai_polish): {len(polish_pairs):,}")

# gen_mode A = ai_fromscratch (ai_degree 4), B/C/D = ai_rewrite family (ai_degree 3)
rewrite_pairs["ai_degree"] = rewrite_pairs["gen_mode"].apply(lambda m: 4 if m == "A" else 3)


# ── CELL 4: Gán split theo split_pool đã có sẵn từ genai.ipynb ───
# train_val_pool -> chia tiếp train/val theo paper_id
# ood_model_pool -> toàn bộ vào test_ood_model
def route_split(row):
    pool = row["split_pool"]

    if pool == "train_val":
        return assign_train_val(row["paper_id"])

    if pool == "ood_model":
        return "test_ood_model"

    return "unknown"


rewrite_pairs["split"] = rewrite_pairs.apply(route_split, axis=1)

print("\nSplit distribution (rewrite pairs):")
print(rewrite_pairs["split"].value_counts().to_string())


# ── CELL 5: test_fp — polish pairs + human dư chưa ghép cặp ──────
used_keys = set(zip(rewrite_pairs["paper_id"], rewrite_pairs["source"])) | \
            set(zip(polish_pairs["paper_id"], polish_pairs["source"]))

not_used_mask = ~human_df.apply(lambda r: (r["paper_id"], r["source"]) in used_keys, axis=1)

# Chi giu leftover thuoc DUNG pool "test_fp" (theo assign_split_pool, khop voi
# genai.ipynb) -- neu khong, paper thuoc pool train_val/ood_model nhung chua
# duoc sinh AI (con pending, hoac sinh that bai) se bi lan nham vao test_fp,
# gay leak paper_id giua 2 split (G1 FAIL). Nhung row nay se bi loai hoan toan
# (xem incomplete_human ben duoi) thay vi bi gan sai split.
is_test_fp_pool = human_df["paper_id"].apply(lambda pid: assign_split_pool(pid) == "test_fp")

leftover_human = human_df[not_used_mask & is_test_fp_pool].copy()
incomplete_human = human_df[not_used_mask & ~is_test_fp_pool]

polish_pairs["split"] = "test_fp"
polish_pairs["ai_degree"] = 1
polish_pairs["label"] = 0  # ground-truth authorship van la human (do FP, khong phai TP)

leftover_human["split"] = "test_fp"
leftover_human["ai_degree"] = 0
leftover_human["ai_text"] = None
leftover_human = leftover_human.rename(columns={"text": "human_text"})
leftover_human["sample_id"] = leftover_human["paper_id"].astype(str) + "_" + leftover_human["source"].astype(str)

print(f"\nPolish pairs cho test_fp : {len(polish_pairs):,}")
print(f"Human dư (chưa từng ghép cặp) cho test_fp: {len(leftover_human):,}")
print(f"Human dư nhưng thuộc pool train_val/ood_model (chưa sinh AI xong hoặc sinh "
      f"thất bại) -- bị LOẠI khỏi assemble, không gán split: {len(incomplete_human):,}")

if len(incomplete_human) > 0:
    print("  -> nếu số này lớn, chạy genai.ipynb thêm (TEST_LIMIT=None) để phủ hết "
          "human_clean.parquet trước khi assemble bản cuối.")


# ── CELL 6: Cân bằng confound (spec §8) — sample_weight theo nhiều feature bề mặt ─
# Trước đây chỉ cân bằng word_count -> diagnostics §12 vẫn bắt shortcut ở
# avg_sentence_len (AUC ~0.56) và punct (đã khử phần lớn ở §7). Giờ cân bằng
# ĐỒNG THỜI trên tích Descartes của các bin (word_count × avg_sentence_len):
# trong mỗi ô lưới, lớp nào ít hơn thì tăng weight để 2 lớp cân nhau, nên detector
# không thể dựa vào các feature này để tách lớp.
def add_sample_weight(df, label_col="label", feat_cols=("word_count", "avg_sentence_len"),
                      n_bins=6):
    df = df.copy()
    df["sample_weight"] = 1.0

    if df[label_col].nunique() < 2:
        return df

    bin_cols = []

    for col in feat_cols:
        if col not in df.columns:
            continue

        try:
            df[f"_bin_{col}"] = pd.qcut(df[col], q=n_bins, duplicates="drop")
            bin_cols.append(f"_bin_{col}")
        except ValueError:
            continue

    if not bin_cols:
        return df

    for _, sub in df.groupby(bin_cols, observed=True):
        n0 = (sub[label_col] == 0).sum()
        n1 = (sub[label_col] == 1).sum()

        if n0 == 0 or n1 == 0:
            continue  # ô không có cả 2 lớp -> reweight không giúp gì, giữ weight 1.0

        # can bang trong tung o luoi: lop nao it hon thi tang weight len
        target = (n0 + n1) / 2
        df.loc[sub.index[sub[label_col] == 0], "sample_weight"] = target / n0
        df.loc[sub.index[sub[label_col] == 1], "sample_weight"] = target / n1

    return df.drop(columns=bin_cols)


train_mask = rewrite_pairs["split"] == "train"
train_for_weight = rewrite_pairs[train_mask].copy()

# Dung tam 1 df long human-vs-ai (chi de tinh weight theo confound, khong ghi ra).
# Feature can bang: word_count + avg_sentence_len (2 confound diagnostics §12 bat).
tmp_human = train_for_weight[["sample_id", "word_count_human", "human_text"]].rename(
    columns={"word_count_human": "word_count", "human_text": "text"}
)
tmp_human["label"] = 0
tmp_ai = train_for_weight[["sample_id", "word_count", "ai_text"]].rename(
    columns={"ai_text": "text"}
)
tmp_ai["label"] = 1
tmp_long = pd.concat([tmp_human, tmp_ai], ignore_index=True)
tmp_long["avg_sentence_len"] = tmp_long["text"].apply(avg_sentence_len)
tmp_long = add_sample_weight(tmp_long, feat_cols=("word_count", "avg_sentence_len"))

weight_by_sample = tmp_long.groupby("sample_id")["sample_weight"].mean()
rewrite_pairs["sample_weight"] = rewrite_pairs["sample_id"].map(weight_by_sample).fillna(1.0)
polish_pairs["sample_weight"] = 1.0
leftover_human["sample_weight"] = 1.0

print("\nsample_weight (train) describe:")
print(rewrite_pairs.loc[train_mask, "sample_weight"].describe().to_string())


# ── CELL 7: QA gate G1 (leakage) / G4 (family quota) / G5 (length parity) ─
print("\n" + "=" * 60)
print("QA GATES")
print("=" * 60)

all_split_frames = [rewrite_pairs, polish_pairs, leftover_human]
paper_split_map = {}
leak_papers = set()

for df in all_split_frames:
    for pid, split in zip(df["paper_id"], df["split"]):
        prev = paper_split_map.get(pid)

        if prev is not None and prev != split:
            leak_papers.add(pid)

        paper_split_map[pid] = split

g1_pass = len(leak_papers) == 0
print(f"G1 (leakage paper_id giữa split = 0) : {'✅ PASS' if g1_pass else '❌ FAIL'}"
      f"  ({len(leak_papers)} paper vi phạm)")

train_rows = rewrite_pairs[rewrite_pairs["split"] == "train"]
family_share = train_rows["generator_family"].value_counts(normalize=True)
g4_pass = bool((family_share <= TRAIN_FAMILY_QUOTA + 0.02).all())  # +2% dung sai lam tron

print(f"G4 (không family nào > {TRAIN_FAMILY_QUOTA:.0%} khối lượng train) : "
      f"{'✅ PASS' if g4_pass else '⚠️  CẢNH BÁO'}")
print(family_share.round(3).to_string())

wc0 = train_rows["word_count_human"]
wc1 = train_rows["word_count"]
mean_diff = abs(wc0.mean() - wc1.mean())

try:
    from scipy.stats import ks_2samp
    ks_stat, ks_p = ks_2samp(wc0, wc1)
    print(f"G5 (length parity) KS-stat={ks_stat:.4f}  p={ks_p:.4f}  "
          f"mean_diff={mean_diff:.1f} từ  ({'⚠️  lệch đáng kể' if ks_p < 0.01 else '✅ OK'})")
except ImportError:
    print(f"G5 (length parity) scipy chưa cài — chỉ log mean_diff={mean_diff:.1f} từ "
          f"({'⚠️  lệch đáng kể' if mean_diff > 20 else '✅ OK'})")

if not g1_pass:
    raise SystemExit("❌ G1 FAIL (leakage) — dừng lại, không ghi output. Kiểm tra assign_split_pool/assign_train_val.")


# ── CELL 8: Ghi output ─────────────────────────────────────────────
PAIR_COLS = [
    "sample_id", "paper_id", "source", "human_text", "ai_text", "word_count",
    "generator_family", "model_actual", "provider", "gen_mode", "prompt_id",
    "temperature", "ai_degree", "sample_weight", "split",
]

for split_name in ["train", "val", "test_ood_model"]:
    sub = rewrite_pairs[rewrite_pairs["split"] == split_name][PAIR_COLS]
    out_path = OUTPUT_DIR / f"dataset_{split_name}_normalized.parquet"
    sub.to_parquet(out_path, index=False)
    print(f"✅ {out_path.name:<40} {len(sub):>7,} dòng")

TEST_FP_PAIR_COLS = [
    "sample_id", "paper_id", "source", "human_text", "ai_text", "ai_degree",
    "sample_weight", "split",
]
test_fp_polish = polish_pairs.reindex(columns=TEST_FP_PAIR_COLS)
test_fp_human = leftover_human.reindex(columns=TEST_FP_PAIR_COLS)
test_fp_all = pd.concat([test_fp_polish, test_fp_human], ignore_index=True)

out_path = OUTPUT_DIR / "dataset_test_fp_normalized.parquet"
test_fp_all.to_parquet(out_path, index=False)
print(f"✅ {out_path.name:<40} {len(test_fp_all):>7,} dòng "
      f"(polish={len(test_fp_polish):,}, human-only={len(test_fp_human):,})")

print("\nLưu ý: dataset_test_fp_normalized.parquet KHÔNG đi qua attack-dataset.ipynb/"
      "paraphrase-attack.ipynb/llm-attack.ipynb (test_fp đo false-positive, không phải "
      "robustness) — đi thẳng tới 04_melt_for_training.ipynb.")
